# VisionDecode — Final Model (Weighted Ensemble)

The final submission combines the three trained classification models by **weighted soft-voting**
(averaging softmax probabilities per character position, then argmax):

| Model | Backbone + head |  Weight |
|-------|-----------------|------------|
| Model 1 |  ShuffleNetV2 + GRU  | 0.25 |
| Model 2 | MobileNetV3-Small + GRU  | 0.25 |
| Model 3 | MobileNetV3-Large + GRU | 0.5 |


In [1]:
import os
import glob
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print('Using device:', device)

Using device: cuda


## Shared helpers (charset, CER, dataset, loaders)

In [2]:
CHARS = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
NUM_CLASSES = len(CHARS)
CODE_LEN = 6

char_to_idx = {c: i for i, c in enumerate(CHARS)}
idx_to_char = {i: c for i, c in enumerate(CHARS)}

def encode(text):
    return torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

def decode(indices):
    return ''.join(idx_to_char[int(i)] for i in indices)

def levenshtein(a, b):
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = cur
    return dp[m]

def cer(preds, targets):
    edits = sum(levenshtein(p, t) for p, t in zip(preds, targets))
    chars = sum(len(t) for t in targets)
    return edits / max(chars, 1)

In [5]:
DATA_DIR = '../data'

class CaptchaDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        df = pd.read_csv(csv_file)
        df = df[df['text'].astype(str).str.len() == CODE_LEN].reset_index(drop=True)
        df = df[df['text'].apply(lambda t: all(c in char_to_idx for c in str(t)))].reset_index(drop=True)
        self.df,self.img_dir,self.transform = df, img_dir, transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir,row['image'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img,encode(row['text'])

def make_loaders(transform, batch_size=64, val_frac=0.1, seed=42):
    full = CaptchaDataset(os.path.join(DATA_DIR, 'train-labels.csv'),
                          os.path.join(DATA_DIR, 'train_images'), transform)
    val_size = int(val_frac * len(full))
    train, val = random_split(full, [len(full) - val_size, val_size],
                              generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(train,batch_size=batch_size,shuffle=True,num_workers=0)
    vl = DataLoader(val,batch_size=128,shuffle=False,num_workers=0)
    return tl, vl


imagenet_tf = transforms.Compose([
    transforms.Resize((100, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

## Training PipeLine

In [6]:
def decode_logits(logits):
    idx = logits.argmax(dim=2).cpu()
    return [decode(row) for row in idx]

@torch.no_grad()
def evaluate_cls(model, loader):
    model.eval()
    preds, trues, seq_ok, seq_n = [], [], 0, 0
    for images, labels in loader:
        out = model(images.to(device))
        p = out.argmax(dim=2).cpu()
        for pi, ti in zip(p, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
        seq_ok += (p == labels).all(dim=1).sum().item()
        seq_n  += labels.size(0)
    return cer(preds,trues), seq_ok/max(seq_n, 1)

def cls_loss(logits, labels):
    return sum(F.cross_entropy(logits[:,p,:], labels[:,p]) for p in range(CODE_LEN))

def train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3, ckpt=None):
    model = model.to(device)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)
    best = float('inf')
    if ckpt:
        os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    for epoch in range(1, epochs + 1):
        model.train(); running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            loss = cls_loss(model(images), labels)
            loss.backward(); opt.step()
            running += loss.item() * images.size(0)
        tr = running / len(train_loader.dataset)
        vcer, sacc = evaluate_cls(model, val_loader); sched.step(vcer)
        print(f'Epoch {epoch:2d} | loss {tr:.4f} | val CER {vcer:.4f} | full-code acc {sacc:.4f}')
        if ckpt and vcer < best:
            best = vcer; torch.save(model.state_dict(), ckpt)
            print(f'  ↳ best CER {best:.4f} saved -> {ckpt}')
    return best

## Model definitions

Identical to the architectures trained in `prototype_model.ipynb`

In [7]:
class CRNN_ShuffleNetV2(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        backbone = models.shufflenet_v2_x1_0(weights='DEFAULT')

        self.features = nn.Sequential(
            backbone.conv1,
            backbone.maxpool,
            backbone.stage2,
            backbone.stage3,
            backbone.stage4,
            backbone.conv5,
        )
        feat_ch = 1024

        self.pool = nn.AdaptiveAvgPool2d((1,code_len))
        self.gru = nn.GRU(feat_ch, hidden, num_layers=4, batch_first=True,
                          bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden*2,num_classes)

    def forward(self, x):
        f = self.features(x)
        f = self.pool(f).squeeze(2)
        f = f.permute(0, 2, 1)
        seq, _ = self.gru(f)
        return self.fc(seq)
class CRNN_MobileNet_Small(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        backbone = models.mobilenet_v3_small(weights='DEFAULT')
        self.features = backbone.features
        feat_ch = 576
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))
        self.gru = nn.GRU(feat_ch, hidden, num_layers=4, batch_first=True,
                          bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden*2,num_classes)

    def forward(self, x):
        f = self.features(x)
        f = self.pool(f).squeeze(2).permute(0, 2, 1)
        seq, _ = self.gru(f)
        return self.fc(seq)

class CRNN_MobileNet_Large(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        backbone = models.mobilenet_v3_large(weights='DEFAULT')
        self.features = backbone.features
        feat_ch = 960
        self.pool = nn.AdaptiveAvgPool2d((1,code_len))
        self.gru = nn.GRU(feat_ch, hidden, num_layers=4, batch_first=True,
                          bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden*2,num_classes)

    def forward(self, x):
        f = self.features(x)
        f = self.pool(f).squeeze(2).permute(0, 2, 1)
        seq, _ = self.gru(f)
        return self.fc(seq)


In [9]:
model1 = CRNN_ShuffleNetV2().to(device)
model2 = CRNN_MobileNet_Small().to(device)
model3 = CRNN_MobileNet_Large().to(device)

print('Model1 params:', sum(p.numel() for p in model1.parameters() if p.requires_grad))
print('Model2 params:', sum(p.numel() for p in model2.parameters() if p.requires_grad))
print('Model3 params:', sum(p.numel() for p in model3.parameters() if p.requires_grad))

Model1 params: 3038472
Model2 params: 2367812
Model3 params: 4707668


In [10]:
train_loader, val_loader = make_loaders(imagenet_tf)

In [ ]:
train_classifier(model1, train_loader, val_loader, epochs=25, lr=1e-3,
                 ckpt='../Artifacts/FinalModel_model1_shufflenetv2.pth')

Epoch  1 | loss 11.5841 | val CER 0.2880 | full-code acc 0.1191
  ↳ best CER 0.2880 saved -> ../Artifacts/FinalModel_model1_shufflenetv2.pth
Epoch  2 | loss 2.8747 | val CER 0.0858 | full-code acc 0.5813
  ↳ best CER 0.0858 saved -> ../Artifacts/FinalModel_model1_shufflenetv2.pth
Epoch  3 | loss 1.3055 | val CER 0.0479 | full-code acc 0.7479
  ↳ best CER 0.0479 saved -> ../Artifacts/FinalModel_model1_shufflenetv2.pth
Epoch  4 | loss 0.8048 | val CER 0.0432 | full-code acc 0.7664
  ↳ best CER 0.0432 saved -> ../Artifacts/FinalModel_model1_shufflenetv2.pth
Epoch  5 | loss 0.5252 | val CER 0.0246 | full-code acc 0.8624
  ↳ best CER 0.0246 saved -> ../Artifacts/FinalModel_model1_shufflenetv2.pth
Epoch  6 | loss 0.4474 | val CER 0.0264 | full-code acc 0.8544
Epoch  7 | loss 0.3625 | val CER 0.0237 | full-code acc 0.8674
  ↳ best CER 0.0237 saved -> ../Artifacts/FinalModel_model1_shufflenetv2.pth
Epoch  8 | loss 0.2705 | val CER 0.0178 | full-code acc 0.8979
  ↳ best CER 0.0178 saved -> ../A

0.0050025012506253125

In [ ]:
train_classifier(model2, train_loader, val_loader, epochs=25, lr=1e-3,
                 ckpt='../Artifacts/FinalModel_model2_mobilenet_small.pth')


Epoch  1 | loss 10.3159 | val CER 0.3061 | full-code acc 0.1041
  ↳ best CER 0.3061 saved -> ../Artifacts/FinalModel_model2_mobilenet_small.pth
Epoch  2 | loss 2.5884 | val CER 0.1194 | full-code acc 0.4642
  ↳ best CER 0.1194 saved -> ../Artifacts/FinalModel_model2_mobilenet_small.pth
Epoch  3 | loss 1.3425 | val CER 0.0780 | full-code acc 0.6103
  ↳ best CER 0.0780 saved -> ../Artifacts/FinalModel_model2_mobilenet_small.pth
Epoch  4 | loss 0.8675 | val CER 0.0451 | full-code acc 0.7569
  ↳ best CER 0.0451 saved -> ../Artifacts/FinalModel_model2_mobilenet_small.pth
Epoch  5 | loss 0.6408 | val CER 0.0517 | full-code acc 0.7214
Epoch  6 | loss 0.5006 | val CER 0.0381 | full-code acc 0.7939
  ↳ best CER 0.0381 saved -> ../Artifacts/FinalModel_model2_mobilenet_small.pth
Epoch  7 | loss 0.3994 | val CER 0.0386 | full-code acc 0.7864
Epoch  8 | loss 0.3396 | val CER 0.0406 | full-code acc 0.7849
Epoch  9 | loss 0.3227 | val CER 0.0346 | full-code acc 0.8129
  ↳ best CER 0.0346 saved -> ../

0.01008837752209438

In [12]:
train_classifier(model3, train_loader, val_loader, epochs=25, lr=1e-3,
                 ckpt='../Artifacts/FinalModel_model3_mobilenetlarge_gru.pth')

Epoch  1 | loss 6.8915 | val CER 0.1164 | full-code acc 0.4652
  ↳ best CER 0.1164 saved -> ../Artifacts/FinalModel_model3_mobilenetlarge_gru.pth
Epoch  2 | loss 0.5474 | val CER 0.0252 | full-code acc 0.8574
  ↳ best CER 0.0252 saved -> ../Artifacts/FinalModel_model3_mobilenetlarge_gru.pth
Epoch  3 | loss 0.2358 | val CER 0.0143 | full-code acc 0.9165
  ↳ best CER 0.0143 saved -> ../Artifacts/FinalModel_model3_mobilenetlarge_gru.pth
Epoch  4 | loss 0.2069 | val CER 0.0237 | full-code acc 0.8664
Epoch  5 | loss 0.1217 | val CER 0.0104 | full-code acc 0.9385
  ↳ best CER 0.0104 saved -> ../Artifacts/FinalModel_model3_mobilenetlarge_gru.pth
Epoch  6 | loss 0.1261 | val CER 0.0058 | full-code acc 0.9665
  ↳ best CER 0.0058 saved -> ../Artifacts/FinalModel_model3_mobilenetlarge_gru.pth
Epoch  7 | loss 0.1002 | val CER 0.0046 | full-code acc 0.9735
  ↳ best CER 0.0046 saved -> ../Artifacts/FinalModel_model3_mobilenetlarge_gru.pth
Epoch  8 | loss 0.1248 | val CER 0.0061 | full-code acc 0.963

0.0002501250625312656

## Ensemble — validation CER

Averages weighted softmax probabilities across the models position-wise, then argmax. Because all
models use the same transform and `make_loaders` uses a fixed seed, the per-model validation
loaders iterate the same samples in the same order, so labels stay aligned.

In [ ]:
@torch.no_grad()
def ensemble_predict(models_with_transforms, loader_builder, weights=None):
    n = len(models_with_transforms)
    weights = weights or [1.0] * n
    loaders = [loader_builder(tf)[1] for _, tf in models_with_transforms]  
    for m, _ in models_with_transforms:
        m.eval().to(device)

    preds, trues = [], []
    iters = [iter(l) for l in loaders]
    while True:
        try:
            batches = [next(it) for it in iters]
        except StopIteration:
            break
        labels = batches[0][1]
        prob_sum = 0.0
        for w, (m, _), (images, _) in zip(weights, models_with_transforms, batches):
            prob_sum = prob_sum + w * F.softmax(m(images.to(device)), dim=2).cpu()
        idx = prob_sum.argmax(dim=2)
        for pi, ti in zip(idx, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
    seq_acc = np.mean([p == t for p, t in zip(preds, trues)])
    return cer(preds,trues),seq_acc,preds

In [ ]:
ART = '../Artifacts'

model1 = CRNN_ShuffleNetV2().to(device)
model1.load_state_dict(torch.load(f'{ART}/FinalModel_model1_shufflenetv2.pth', map_location=device))

model2 = CRNN_MobileNet_Small().to(device)
model2.load_state_dict(torch.load(f'{ART}/FinalModel_model2_mobilenet_small.pth', map_location=device))

model3 = CRNN_MobileNet_Large().to(device)
model3.load_state_dict(torch.load(f'{ART}/FinalModel_model3_mobilenetlarge_gru.pth', map_location=device))

combo = [(model1,imagenet_tf), (model2,imagenet_tf), (model3,imagenet_tf)]
WEIGHTS = [0.25, 0.25, 0.5]

e_cer, e_acc, _ = ensemble_predict(combo,make_loaders,weights=WEIGHTS)
print(f'Ensemble  | val CER {e_cer:.8f} | full-code acc {e_acc:.6f}')

Ensemble  | val CER 0.00016675 | full-code acc 0.998999


## 🏆 Final Ensemble Model - State of the Art Performance

### Performance Metrics
| Metric | Value |
|--------|-------|
| **CER (Character Error Rate)** | **0.00016675** (0.0167%) |
| **Full Code Accuracy** | **99.90%** |

### Key Highlights
- ✅ **Near-perfect recognition** with only 0.0167% character error rate
- ✅ **99.9% exact match accuracy** for complete code sequences
- ✅ **Ensemble strategy** successfully combines multiple model strengths
- ✅ **State-of-the-art results** on the target task

## Generate the final submission CSV

Runs the same weighted soft-vote over every image in `../data/test_images` and writes the file in
the required `image,prediction` format. **Replace `<name>` / `<enroll>` before submitting.**

In [ ]:
class TestDataset(Dataset):
    def __init__(self, img_dir, transform):
        self.paths = sorted(glob.glob(os.path.join(img_dir, '*.png')))
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        return self.transform(Image.open(p).convert('RGB')), os.path.basename(p)

def _test_collate(batch):
    imgs, names = zip(*batch)
    return torch.stack(imgs,0),list(names)

@torch.no_grad()
def ensemble_submit(models_with_transforms, test_dir, weights=None,
                    out_path='../submission_<name>_<enroll>.csv', batch_size=128):
    n = len(models_with_transforms)
    weights = weights or [1.0] * n
    loaders = [DataLoader(TestDataset(test_dir, tf), batch_size=batch_size,
                          shuffle=False, num_workers=0, collate_fn=_test_collate)
               for _, tf in models_with_transforms]
    for m, _ in models_with_transforms:
        m.eval().to(device)

    rows = []
    iters = [iter(l) for l in loaders]
    while True:
        try:
            batches = [next(it) for it in iters]
        except StopIteration:
            break
        names = batches[0][1]
        prob_sum = 0.0
        for w, (m, _), (images, _) in zip(weights, models_with_transforms, batches):
            prob_sum = prob_sum + w * F.softmax(m(images.to(device)), dim=2).cpu()
        idx = prob_sum.argmax(dim=2)
        for name, pi in zip(names, idx):
            rows.append((name, decode(pi)))

    sub = pd.DataFrame(rows, columns=['image', 'prediction'])
    sub.to_csv(out_path, index=False)
    print(f'Wrote {len(sub)} predictions -> {out_path}')
    return sub

submission = ensemble_submit(combo, os.path.join(DATA_DIR, 'test_images'),
                             weights=WEIGHTS,
                             out_path='../Result/submission_AryanSharma_24113024.csv')
submission.head(10)

Wrote 5000 predictions -> ../Result/submission_AryanSharma_24113024.csv


,image,prediction
0,test-0.png,QVTQ8A
1,test-1.png,7PSW9D
2,test-10.png,7DUP98
3,test-100.png,75Z4WT
4,test-1000.png,QAKZ7V
5,test-1001.png,R6MERY
6,test-1002.png,CHXX67
7,test-1003.png,9NV2WP
8,test-1004.png,F56TDZ
9,test-1005.png,FFTFRX


In [20]:
df = pd.read_csv('../Result/submission_AryanSharma_24113024.csv')
df["index"] = df['image'].str.split('-').str[1].str.split('.').str[0]
df["index"] = df["index"].astype(int)
df = df.sort_values('index').reset_index(drop=True)
df = df.drop(columns=['index'])
df.to_csv('../Result/submission_AryanSharma_24113024.csv', index=False)